# Paper 1 — Golden Age Semantic Reconfiguration

**Working title:** *Reconfiguring the Golden Age: Semantic Networks and the Renaissance–Baroque Transition in Spanish Poetry*

> **Current stage: Phase 4 — Scholarly Chronology Acquisition (robust rebuild).**

This notebook rebuilds the Phase 4 workflow cleanly after diagnosing two parsing failures. The key correction is structural: Navarro's TEI header contains a general corpus `<title>`, while the poem title (e.g. `-I-`) lives inside `<text><body><head><title>`. We therefore extract the **body-level poem title**, and use the source filename as an independent numbering cross-check for Garcilaso.

We still **do not** build semantic networks or choose temporal windows.


In [26]:
import sys, re, shutil, subprocess, unicodedata
from pathlib import Path
from difflib import SequenceMatcher
import pandas as pd
import xml.etree.ElementTree as ET

SOURCES = {
    "navarro_tei": (
        "https://github.com/bncolorado/CorpusSonetosSigloDeOro.git",
        "092a5fe70a4065a4d84bfed288bffd3851348f9c",
    ),
    "gongora_scholarly": (
        "https://github.com/gongoradigital/gongoraobra.git",
        "3beadeecc059a7cc48499dc2683bb378a2630978",
    ),
}

ROOT = Path("/content/gasr_phase4_sources")
ROOT.mkdir(exist_ok=True)

def clone(name, url, commit):
    dst = ROOT / name
    if dst.exists():
        shutil.rmtree(dst)
    subprocess.run(["git","clone","--quiet",url,str(dst)], check=True)
    subprocess.run(["git","-C",str(dst),"checkout","--quiet",commit], check=True)
    got = subprocess.check_output(["git","-C",str(dst),"rev-parse","HEAD"], text=True).strip()
    assert got == commit, (name, got, commit)
    return dst

paths = {k: clone(k, *v) for k,v in SOURCES.items()}
N = paths["navarro_tei"]
G = paths["gongora_scholarly"]
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"

def local(tag):
    return tag.split("}")[-1] if "}" in tag else tag

def el_text(el):
    return "" if el is None else " ".join(" ".join(el.itertext()).split())

def norm(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", s.lower())

def years_1580_1626(s):
    return sorted(set(
        int(x) for x in re.findall(r"(?<!\d)(1[56]\d{2})(?!\d)", str(s))
        if 1580 <= int(x) <= 1626
    ))

print("Pinned sources ready")
for k,v in SOURCES.items():
    print(f"  {k}: {v[1]}")
print("Python", sys.version.split()[0], "| pandas", pd.__version__)


Pinned sources ready
  navarro_tei: 092a5fe70a4065a4d84bfed288bffd3851348f9c
  gongora_scholarly: 3beadeecc059a7cc48499dc2683bb378a2630978
Python 3.13.15 | pandas 2.2.3


## 01. Rebuild the Navarro poem backbone

The earlier failure came from reading the first `<title>` in each TEI file, which is the generic header title *Spanish Metrical Patterns Bank: Golden Age Sonnets.* The poem title is stored under the TEI `<body>`. This cell extracts it structurally.

No witness, edition, or publication year is promoted to composition time.


In [27]:
def body_poem_title(root):
    """Return the first title inside TEI <body>; never use the header title."""
    for body in root.iter():
        if local(body.tag) == "body":
            for x in body.iter():
                if local(x.tag) == "title":
                    t = el_text(x)
                    if t:
                        return t
            break
    return ""

rows, parse_errors = [], []
xml_files = sorted(N.rglob("*.xml"))

for fp in xml_files:
    try:
        root = ET.parse(fp).getroot()
    except Exception as e:
        parse_errors.append((str(fp), repr(e)))
        continue

    lines = [el_text(x) for x in root.iter() if local(x.tag) == "l"]
    lines = [x for x in lines if x]
    if not lines:
        continue

    author_dir = fp.parent.name
    text = "\n".join(lines)
    poem_title = body_poem_title(root)
    bibls = [el_text(x) for x in root.iter()
             if local(x.tag) in {"bibl","witness"} and el_text(x)]

    rows.append({
        "n_id": f"{author_dir}::{fp.name}",
        "author_dir": author_dir,
        "title": poem_title,
        "n_lines": len(lines),
        "text": text,
        "signature": norm(text),
        "source_bibl": " | ".join(bibls[:4]),
        "source_file": str(fp.relative_to(N)),
    })

n = pd.DataFrame(rows)
assert len(parse_errors) == 0, parse_errors[:5]
assert len(n) == 5078, f"Expected 5,078 Navarro poem records, found {len(n)}"

priority_A = {
    "GarcilasoDeLaVega","JuanBoscan","FernandoDeHerrera","PedroEspinosa",
    "JuanDeArguijo","JuanDeJauregui","LuisCarrilloySotomayor","Cervantes",
    "Gongora","LopeDeVega_1","LopeDeVega_2","Quevedo"
}

print(f"Navarro poems: {len(n):,} | author folders: {n.author_dir.nunique():,}")
print("Garcilaso first five body-level poem titles:")
display(n[n.author_dir.eq("GarcilasoDeLaVega")][["n_id","title"]].head())
print("Priority-A poem counts:")
display(
    n[n.author_dir.isin(priority_A)]
    .groupby("author_dir").size()
    .sort_values(ascending=False)
    .rename("poems").reset_index()
)


Navarro poems: 5,078 | author folders: 53
Garcilaso first five body-level poem titles:


,n_id,title
1356,GarcilasoDeLaVega::GarcilasoDeLaVega_01.xml,-I-
1357,GarcilasoDeLaVega::GarcilasoDeLaVega_02.xml,-II-
1358,GarcilasoDeLaVega::GarcilasoDeLaVega_03.xml,-III-
1359,GarcilasoDeLaVega::GarcilasoDeLaVega_04.xml,-IV-
1360,GarcilasoDeLaVega::GarcilasoDeLaVega_05.xml,-V-


Priority-A poem counts:


,author_dir,poems
0,LopeDeVega_1,699
1,LopeDeVega_2,647
2,Quevedo,517
3,FernandoDeHerrera,320
4,Gongora,115
5,JuanBoscan,100
6,Cervantes,77
7,JuanDeArguijo,70
8,LuisCarrilloySotomayor,50
9,GarcilasoDeLaVega,38


## 02. Temporal master candidate

Temporal evidence is recorded explicitly and conservatively.

- **A**: strong poem-level scholarly year / historical anchor.
- **B**: defensible scholarly interval or very high-confidence linkage.
- **C**: broad or contested interval requiring sensitivity analysis.
- **unassigned**: no defensible composition-time evidence yet.

The primary axis is composition/scholarly chronology. Publication, witness, and edition dates remain secondary evidence only.


In [28]:
temporal = n[["n_id","author_dir","title","source_file"]].copy()
for c in ["composition_min","composition_max","circulation_year"]:
    temporal[c] = pd.NA
temporal["temporal_confidence"] = "unassigned"
temporal["temporal_basis"] = ""
temporal["temporal_source"] = ""
temporal["chronology_status"] = "undated"

def assign_date_scalar(n_ids, lo, hi, confidence, basis, source):
    ids = set(n_ids)
    mask = temporal.n_id.isin(ids)
    if mask.sum() != len(ids):
        missing = ids - set(temporal.loc[mask,"n_id"])
        raise ValueError(f"Temporal assignment IDs not found: {sorted(missing)[:5]}")
    if not (temporal.loc[mask,"chronology_status"] == "undated").all():
        overlap = temporal.loc[mask & temporal.chronology_status.ne("undated"), "n_id"].tolist()
        raise ValueError(f"Temporal assignment collision: {overlap[:5]}")
    temporal.loc[mask,"composition_min"] = int(lo)
    temporal.loc[mask,"composition_max"] = int(hi)
    temporal.loc[mask,"temporal_confidence"] = confidence
    temporal.loc[mask,"temporal_basis"] = basis
    temporal.loc[mask,"temporal_source"] = source
    temporal.loc[mask,"chronology_status"] = "dated_current_sprint"

print("Temporal schema initialized:", len(temporal), "poems")


Temporal schema initialized: 5078 poems


## 03. Góngora: scholarly chronology and poem linkage

The Cátedra Góngora XML is used as an independent scholarly chronology source. Years are extracted from the local structural context of each poem division. Navarro sonnets are then linked by normalized text; exact matches are preferred, and fuzzy matches at ≥0.98 are accepted only when the target is one-to-one.

The extracted year is labelled `scholarly_chronology_year`, not documentary composition date.


In [29]:
gfile = G / "gongora_obra-poetica.xml"
assert gfile.exists(), gfile
groot = ET.parse(gfile).getroot()
parent = {child: par for par in groot.iter() for child in par}

poem_divs = []
for el in groot.iter():
    xid = el.attrib.get(XML_ID,"")
    if local(el.tag) == "div" and xid.lower().startswith("poem"):
        ls = [x for x in el.iter() if local(x.tag) == "l"]
        if ls:
            poem_divs.append(el)

def shallow_years(el):
    vals = list(el.attrib.values())
    if el.text:
        vals.append(el.text)
    for ch in list(el):
        if local(ch.tag) in {"head","date","label"}:
            vals.append(el_text(ch))
        if ch.tail:
            vals.append(ch.tail)
    ys = []
    for v in vals:
        ys.extend(years_1580_1626(v))
    return ys

def ancestor_years(el, max_steps=6):
    ys, cur = [], el
    for _ in range(max_steps):
        ys.extend(shallow_years(cur))
        cur = parent.get(cur)
        if cur is None:
            break
    return sorted(set(ys))

grows = []
for el in poem_divs:
    ls = [el_text(x) for x in el.iter() if local(x.tag) == "l"]
    ls = [x for x in ls if x]
    ys = ancestor_years(el)
    grows.append({
        "g_id": el.attrib.get(XML_ID,""),
        "g_n": el.attrib.get("n",""),
        "n_lines": len(ls),
        "text": "\n".join(ls),
        "signature": norm("\n".join(ls)),
        "year_candidates": ";".join(map(str,ys)),
        "scholarly_year": ys[0] if len(ys) == 1 else pd.NA,
        "year_status": "unique" if len(ys) == 1 else ("ambiguous" if len(ys) > 1 else "missing"),
    })

g = pd.DataFrame(grows)
print(f"Góngora XML poem divisions: {len(g):,}")
print("Year extraction status:")
display(g.year_status.value_counts().rename_axis("status").reset_index(name="poems"))
print(
    "Chronology range among uniquely resolved years:",
    g.scholarly_year.dropna().min(), "–", g.scholarly_year.dropna().max()
)
print("14-line scholarly poems:", int((g.n_lines == 14).sum()))


Góngora XML poem divisions: 481
Year extraction status:


,status,poems
0,unique,471
1,missing,9
2,ambiguous,1


Chronology range among uniquely resolved years: 1580 – 1626
14-line scholarly poems: 192


In [30]:
ng = n[n.author_dir.eq("Gongora")].copy()
g14 = g[g.n_lines.eq(14) & g.signature.ne("")].copy()

sig_to_gids = g14.groupby("signature").g_id.apply(list).to_dict()
g_by_id = g.set_index("g_id", drop=False)

link_rows = []
for r in ng.itertuples(index=False):
    exact_ids = sig_to_gids.get(r.signature, [])
    if len(exact_ids) == 1:
        gid = exact_ids[0]
        score = 1.0
        method = "exact"
    else:
        best_gid, best_score = None, -1.0
        for gr in g14.itertuples(index=False):
            sc = SequenceMatcher(None, r.signature, gr.signature).ratio()
            if sc > best_score:
                best_score = sc
                best_gid = gr.g_id
        gid = best_gid
        score = best_score
        method = "fuzzy_provisional"

    link_rows.append({
        "n_id": r.n_id,
        "g_id": gid,
        "method": method,
        "score": score,
        "preaccept": (method == "exact") or (score >= 0.98),
    })

glink = pd.DataFrame(link_rows)
candidate = glink[glink.preaccept].copy()
collision_ids = set(
    candidate.groupby("g_id").size().loc[lambda s: s > 1].index
)
glink["accept"] = glink.preaccept & ~glink.g_id.isin(collision_ids)

# Attach unique scholarly years.
glink["scholarly_year"] = glink.g_id.map(g_by_id.scholarly_year.to_dict())
accepted_dated = glink[glink.accept & glink.scholarly_year.notna()].copy()

for r in accepted_dated.itertuples(index=False):
    conf = "A" if r.method == "exact" else "B"
    assign_date_scalar(
        [r.n_id], int(r.scholarly_year), int(r.scholarly_year),
        conf, "scholarly_chronology_year",
        "Cátedra Góngora / Carreira chronology; XML-TEI gongoradigital/gongoraobra"
    )

print("Navarro Góngora sonnets:", len(ng))
print("Exact links:", int((glink.method == "exact").sum()))
print("Fuzzy >= .98 candidates:", int(((glink.method == "fuzzy_provisional") & glink.preaccept).sum()))
print("Accepted one-to-one links:", int(glink.accept.sum()))
print("Accepted links with unique scholarly year:", len(accepted_dated))
print("Colliding scholarly targets excluded:", len(collision_ids))
display(
    glink.assign(score=lambda d: d.score.round(4))
    .sort_values(["accept","score"], ascending=[False,False])
    .head(20)
)


Navarro Góngora sonnets: 115
Exact links: 12
Fuzzy >= .98 candidates: 41
Accepted one-to-one links: 53
Accepted links with unique scholarly year: 53
Colliding scholarly targets excluded: 0


,n_id,g_id,method,score,preaccept,accept,scholarly_year
13,Gongora::Gongora_110.xml,poem464,exact,1.0000,True,True,1622.0
14,Gongora::Gongora_111.xml,poem460,exact,1.0000,True,True,1618.0
17,Gongora::Gongora_114.xml,poem462,exact,1.0000,True,True,1621.0
43,Gongora::Gongora_34.xml,poem42,exact,1.0000,True,True,1584.0
47,Gongora::Gongora_38.xml,poem18,exact,1.0000,True,True,1582.0
55,Gongora::Gongora_45.xml,poem116,exact,1.0000,True,True,1600.0
58,Gongora::Gongora_48.xml,poem138,exact,1.0000,True,True,1603.0
59,Gongora::Gongora_49.xml,poem155,exact,1.0000,True,True,1604.0
63,Gongora::Gongora_52.xml,poem13,exact,1.0000,True,True,1582.0
64,Gongora::Gongora_53.xml,poem355,exact,1.0000,True,True,1621.0


## 04. Garcilaso: canonical numbering and conservative chronology seed

Navarro's Garcilaso files are named `GarcilasoDeLaVega_01.xml` … `_38.xml`. The body-level titles are `-I-` … `-XXXVIII-`.

The **filename number is the primary machine identifier** because it is explicit and complete. The Roman title is used only as an independent integrity cross-check; a title-parsing anomaly is reported but no longer aborts the entire notebook.


In [31]:
roman_vals = {"I":1,"V":5,"X":10,"L":50,"C":100,"D":500,"M":1000}

def roman_to_int(s):
    if not s:
        return pd.NA
    total, prev = 0, 0
    for ch in reversed(s):
        v = roman_vals.get(ch, 0)
        if v == 0:
            return pd.NA
        total += -v if v < prev else v
        prev = max(prev, v)
    return total

def extract_roman_token(s):
    # Ignore punctuation/dash type and extract the Roman numeral token itself.
    m = re.search(r"(?<![A-Z])([IVXLCDM]+)(?![A-Z])", str(s).upper())
    return m.group(1) if m else ""

gar = n[n.author_dir.eq("GarcilasoDeLaVega")].copy()
gar["file_no"] = pd.to_numeric(
    gar.n_id.str.extract(r"_(\d+)\.xml$", expand=False),
    errors="coerce"
).astype("Int64")
gar["sonnet_roman"] = gar.title.map(extract_roman_token)
gar["title_no"] = gar.sonnet_roman.map(lambda x: roman_to_int(x) if x else pd.NA).astype("Int64")

# The source filename is complete, unique, and ordered 1..38.
assert len(gar) == 38, f"Expected 38 Garcilaso sonnets, found {len(gar)}"
assert gar.file_no.notna().all(), "Some Garcilaso filenames were not parsed."
assert gar.file_no.nunique() == 38, "Duplicate Garcilaso filename numbers."
assert set(gar.file_no.astype(int)) == set(range(1,39)), "Garcilaso filename numbering is not exactly 1..38."

title_parsed = int(gar.title_no.notna().sum())
agreements = int((gar.title_no == gar.file_no).fillna(False).sum())

print("Garcilaso Navarro records:", len(gar))
print("Roman-title numbers parsed:", title_parsed)
print("Filename numbers parsed:", int(gar.file_no.notna().sum()))
print("Title/filename agreements:", agreements)

if title_parsed == 38 and agreements == 38:
    print("Garcilaso numbering integrity: PASSED (38/38).")
else:
    print("WARNING: title-level cross-check incomplete; filename numbering remains the primary canonical key.")
    display(gar.loc[(gar.title_no != gar.file_no).fillna(True), ["n_id","title","title_no","file_no"]])

gar["sonnet_no"] = gar.file_no
display(gar[["n_id","title","sonnet_no"]].sort_values("sonnet_no").head(10))


Garcilaso Navarro records: 38
Roman-title numbers parsed: 38
Filename numbers parsed: 38
Title/filename agreements: 38
Garcilaso numbering integrity: PASSED (38/38).


,n_id,title,sonnet_no
1356,GarcilasoDeLaVega::GarcilasoDeLaVega_01.xml,-I-,1
1357,GarcilasoDeLaVega::GarcilasoDeLaVega_02.xml,-II-,2
1358,GarcilasoDeLaVega::GarcilasoDeLaVega_03.xml,-III-,3
1359,GarcilasoDeLaVega::GarcilasoDeLaVega_04.xml,-IV-,4
1360,GarcilasoDeLaVega::GarcilasoDeLaVega_05.xml,-V-,5
1361,GarcilasoDeLaVega::GarcilasoDeLaVega_06.xml,-VI-,6
1362,GarcilasoDeLaVega::GarcilasoDeLaVega_07.xml,-VII-,7
1363,GarcilasoDeLaVega::GarcilasoDeLaVega_08.xml,-VIII-,8
1364,GarcilasoDeLaVega::GarcilasoDeLaVega_09.xml,-IX-,9
1365,GarcilasoDeLaVega::GarcilasoDeLaVega_10.xml,-X-,10


In [32]:
# Conservative first-pass chronology seed.
# These intervals are not inferred from Navarro metadata; they are entered only
# where prior philological scholarship provides a defensible phase/anchor.
GAR_CHRONOLOGY = {
    **{i:(1526,1532,"B","scholarly_phase_interval") for i in [1,2,3,4,6,26,27]},
    25:(1534,1535,"B","scholarly_interval"),
    33:(1535,1535,"A","historically_anchored_scholarly_year"),
    35:(1535,1535,"A","historically_anchored_scholarly_year"),
    **{i:(1533,1535,"B","revised_scholarly_interval") for i in [7,8,12,15,19,28,30,31]},
}

gassign = []
for no,(lo,hi,conf,basis) in GAR_CHRONOLOGY.items():
    z = gar[gar.sonnet_no.eq(no)]
    if len(z) != 1:
        print("WARNING: canonical number not uniquely found:", no, len(z))
        continue
    nid = z.iloc[0].n_id
    assign_date_scalar(
        [nid], lo, hi, conf, basis,
        "Rafael Lapesa chronology as summarized/discussed by CVC (E. L. Rivers) and AISO scholarship"
    )
    gassign.append({
        "sonnet_no": no,
        "n_id": nid,
        "composition_min": lo,
        "composition_max": hi,
        "temporal_confidence": conf,
        "temporal_basis": basis,
    })

gar_chron = pd.DataFrame(gassign).sort_values("sonnet_no")
print("Garcilaso conservatively dated this sprint:", len(gar_chron), "/", len(gar))
print("Garcilaso still unassigned:", len(gar) - len(gar_chron))
display(gar_chron)


Garcilaso conservatively dated this sprint: 18 / 38
Garcilaso still unassigned: 20


,sonnet_no,n_id,composition_min,composition_max,temporal_confidence,temporal_basis
0,1,GarcilasoDeLaVega::GarcilasoDeLaVega_01.xml,1526,1532,B,scholarly_phase_interval
1,2,GarcilasoDeLaVega::GarcilasoDeLaVega_02.xml,1526,1532,B,scholarly_phase_interval
2,3,GarcilasoDeLaVega::GarcilasoDeLaVega_03.xml,1526,1532,B,scholarly_phase_interval
3,4,GarcilasoDeLaVega::GarcilasoDeLaVega_04.xml,1526,1532,B,scholarly_phase_interval
4,6,GarcilasoDeLaVega::GarcilasoDeLaVega_06.xml,1526,1532,B,scholarly_phase_interval
10,7,GarcilasoDeLaVega::GarcilasoDeLaVega_07.xml,1533,1535,B,revised_scholarly_interval
11,8,GarcilasoDeLaVega::GarcilasoDeLaVega_08.xml,1533,1535,B,revised_scholarly_interval
12,12,GarcilasoDeLaVega::GarcilasoDeLaVega_12.xml,1533,1535,B,revised_scholarly_interval
13,15,GarcilasoDeLaVega::GarcilasoDeLaVega_15.xml,1533,1535,B,revised_scholarly_interval
14,19,GarcilasoDeLaVega::GarcilasoDeLaVega_19.xml,1533,1535,B,revised_scholarly_interval


## 05. Current Priority-A temporal coverage

This table is diagnostic. It tells us how much of the primary composition-time axis is currently defensible after the first two scholarly acquisitions, and which authors require the next philological search.


In [33]:
coverage = (
    n[n.author_dir.isin(priority_A)]
    .groupby("author_dir").size().rename("total_poems").to_frame()
    .join(
        temporal[temporal.chronology_status.ne("undated")]
        .groupby("author_dir").size().rename("dated_current")
    )
    .fillna(0)
)
coverage["dated_current"] = coverage.dated_current.astype(int)
coverage["coverage_pct"] = (100 * coverage.dated_current / coverage.total_poems).round(1)
coverage = coverage.sort_values(
    ["coverage_pct","total_poems"], ascending=[False,False]
).reset_index()

display(coverage)

print("Confidence distribution among currently dated poems:")
display(
    temporal[temporal.chronology_status.ne("undated")]
    .temporal_confidence.value_counts()
    .rename_axis("confidence").reset_index(name="poems")
)

dated = temporal[temporal.chronology_status.ne("undated")].copy()
assert dated.composition_min.notna().all() and dated.composition_max.notna().all()
assert (dated.composition_min.astype(int) <= dated.composition_max.astype(int)).all()
assert not dated.temporal_basis.str.contains(
    "publication|witness|edition", case=False, regex=True
).any(), "Publication/witness/edition evidence leaked into the composition-time axis."

print("Temporal integrity checks: PASSED")


,author_dir,total_poems,dated_current,coverage_pct
0,GarcilasoDeLaVega,38,18,47.4
1,Gongora,115,53,46.1
2,LopeDeVega_1,699,0,0.0
3,LopeDeVega_2,647,0,0.0
4,Quevedo,517,0,0.0
5,FernandoDeHerrera,320,0,0.0
6,JuanBoscan,100,0,0.0
7,Cervantes,77,0,0.0
8,JuanDeArguijo,70,0,0.0
9,LuisCarrilloySotomayor,50,0,0.0


Confidence distribution among currently dated poems:


,confidence,poems
0,B,57
1,A,14


Temporal integrity checks: PASSED


## 06. Runtime exports and checkpoint

Derived CSVs are written only to the Colab runtime. The notebook and pinned source commits remain the provenance record. Stable derived tables can be committed later after review.


In [34]:
OUT = Path("/content/gasr_phase4_outputs")
OUT.mkdir(exist_ok=True)

temporal.to_csv(OUT/"temporal_master_candidate.csv", index=False)
glink.to_csv(OUT/"gongora_match_diagnostics.csv", index=False)
gar_chron.to_csv(OUT/"garcilaso_chronology_seed.csv", index=False)
coverage.to_csv(OUT/"priority_A_temporal_coverage.csv", index=False)

print("Runtime outputs:")
for p in sorted(OUT.glob("*.csv")):
    print(" ", p)

print()
print("PHASE 4 CHECKPOINT")
print("------------------")
print("Save this executed notebook to GitHub.")
print("Do NOT build semantic networks yet.")
print("Next: inspect Góngora scholarly-year linkage and Garcilaso coverage;")
print("then extend defensible dating to the remaining Priority-A authors.")


Runtime outputs:
  /content/gasr_phase4_outputs/garcilaso_chronology_seed.csv
  /content/gasr_phase4_outputs/gongora_match_diagnostics.csv
  /content/gasr_phase4_outputs/priority_A_temporal_coverage.csv
  /content/gasr_phase4_outputs/temporal_master_candidate.csv

PHASE 4 CHECKPOINT
------------------
Save this executed notebook to GitHub.
Do NOT build semantic networks yet.
Next: inspect Góngora scholarly-year linkage and Garcilaso coverage;
then extend defensible dating to the remaining Priority-A authors.
